# AlpCAN RPT-1: Yapilandirilmis Radyoloji Rapor Motoru

**Amac:** AlpCAN pipeline ciktilarindan otomatik yapilandirilmis radyoloji raporu olusturma.
Rule-based sablon sistemi ile Turkce radyoloji raporlari uretir.

**Giris:** Pipeline JSON ciktilari (CXR analiz, CT nodul tespiti, segmentasyon, karakterizasyon, malignite skoru)
**Cikis:** Yapilandirilmis Turkce radyoloji raporu (PDF-ready HTML + JSON + CSV)

**Ozellikler:**
- BI-RADS / Lung-RADS uyumlu raporlama
- ACR (American College of Radiology) sablon yapisi
- Coklu modalite destegi (CXR + CT)
- Bulgularin oncelik siralamasi
- Takip onerileri
- Kalite kontrol entegrasyonu

In [ ]:
# ============================================================
# NB14 — RPT-1 Raporlama Motoru: Kutuphaneler
# ============================================================
import json
import csv
import random
import html as html_module
from datetime import datetime
from pathlib import Path
from dataclasses import dataclass, field, asdict
from enum import Enum
from typing import List, Optional, Dict, Any
from collections import Counter

# Cikti dizini
OUTPUT_DIR = Path('/kaggle/working')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('AlpCAN RPT-1 Raporlama Motoru — Kutuphaneler yuklendi.')
print(f'Cikti dizini: {OUTPUT_DIR}')
print(f'Tarih: {datetime.now().strftime("%d.%m.%Y %H:%M")}')

In [ ]:
# ============================================================
# Veri Yapilari — Enum ve Dataclass Tanimlamalari
# ============================================================

class Modalite(Enum):
    CXR = 'Gogus Rontgeni'
    BT = 'Bilgisayarli Tomografi'


class Aciliyet(Enum):
    NORMAL = 'Normal'
    TAKIP = 'Takip Gerekli'
    ACIL = 'Acil Konsultasyon'
    KRITIK = 'Kritik - Hemen Bildirim'


class LungRADS(Enum):
    CAT_1 = '1'
    CAT_2 = '2'
    CAT_3 = '3'
    CAT_4A = '4A'
    CAT_4B = '4B'
    CAT_4X = '4X'


@dataclass
class Bulgu:
    """Tek bir radyolojik bulgu."""
    id: int
    tip: str            # 'nodul', 'opasite', 'konsolidasyon', vb.
    lokasyon: str        # 'right_upper', 'left_lower', vb.
    boyut_mm: Optional[float] = None
    malignite_skoru: Optional[float] = None
    lung_rads: Optional[str] = None
    guven_skoru: float = 0.0
    ozellikler: Dict = field(default_factory=dict)
    aciklama: str = ''


@dataclass
class CXRAnaliz:
    """CXR pipeline ciktisi."""
    patolojiler: Dict[str, float]   # patoloji adi -> olasilik
    kalite_skoru: float = 0.0
    goruntulenebilirlik: str = 'Yeterli'


@dataclass
class BTAnaliz:
    """BT pipeline ciktisi."""
    noduller: List[Bulgu] = field(default_factory=list)
    toplam_nodul: int = 0
    en_buyuk_nodul_mm: float = 0.0
    genel_lung_rads: str = '1'


@dataclass
class HastaBilgisi:
    hasta_id: str
    hasta_adi: str
    yas: int
    cinsiyet: str
    protokol_no: str
    tarih: str = ''
    klinik_bilgi: str = ''
    onceki_calisma: Optional[str] = None


@dataclass
class RaporSonucu:
    hasta: HastaBilgisi
    modalite: str
    teknik: str
    bulgular: List[str]
    izlenim: List[str]
    lung_rads: Optional[str]
    aciliyet: str
    takip_onerisi: str
    ai_ozet: Dict
    rapor_html: str = ''
    rapor_text: str = ''


print('Veri yapilari tanimlandi:')
print('  - Modalite, Aciliyet, LungRADS (Enum)')
print('  - Bulgu, CXRAnaliz, BTAnaliz, HastaBilgisi, RaporSonucu (dataclass)')

## 2. Rapor Sablon Motoru

ACR (American College of Radiology) uyumlu yapilandirilmis radyoloji rapor motoru.
Pipeline ciktilarini alir, kurallara dayali olarak bulgu ve izlenim cumleleri olusturur.
Turkce terminoloji ve Lung-RADS kategorilendirmesi icerir.

In [ ]:
# ============================================================
# RaporMotoru — Ana Rapor Uretim Sinifi
# ============================================================

class RaporMotoru:
    """AlpCAN Yapilandirilmis Radyoloji Rapor Motoru.

    ACR uyumlu rapor sablonu ile Turkce radyoloji raporlari uretir.
    Pipeline ciktilarini alir, kurallara gore bulgu/izlenim olusturur.
    """

    # Lokasyon cevirileri
    LOKASYON = {
        "right_upper": "Sag akciger ust lob",
        "right_middle": "Sag akciger orta lob",
        "right_lower": "Sag akciger alt lob",
        "left_upper": "Sol akciger ust lob",
        "left_lower": "Sol akciger alt lob",
        "right_hilum": "Sag hilus",
        "left_hilum": "Sol hilus",
        "mediastinum": "Mediastinum",
        "right_pleural": "Sag plevral alan",
        "left_pleural": "Sol plevral alan",
    }

    # CXR patoloji cevirileri
    PATOLOJI = {
        "Atelectasis": "Atelektazi",
        "Cardiomegaly": "Kardiyomegali",
        "Consolidation": "Konsolidasyon",
        "Edema": "Odem",
        "Effusion": "Plevral Efuzyon",
        "Emphysema": "Amfizem",
        "Fibrosis": "Fibrozis",
        "Hernia": "Herni",
        "Infiltration": "Infiltrasyon",
        "Mass": "Kitle",
        "Nodule": "Nodul",
        "Pleural_Thickening": "Plevral Kalinlasma",
        "Pneumonia": "Pnomoni",
        "Pneumothorax": "Pnomotoraks",
        "No Finding": "Normal",
    }

    # Lung-RADS takip onerileri
    LUNGRADS_TAKIP = {
        "1": "Yillik dusuk doz BT taramasi ile takip onerilir.",
        "2": "Yillik dusuk doz BT taramasi ile takip onerilir.",
        "3": "6 ay sonra dusuk doz BT ile kontrol onerilir.",
        "4A": "3 ay sonra dusuk doz BT veya PET/BT onerilir. Klinik korelasyon gereklidir.",
        "4B": "Gogus cerrahisi konsultasyonu ve biyopsi/PET-BT onerilir.",
        "4X": "Acil gogus cerrahisi konsultasyonu onerilir. Biyopsi endikasyonu degerlendirilmelidir.",
    }

    # Lung-RADS aciliyet esleme
    LUNGRADS_ACILIYET = {
        "1": Aciliyet.NORMAL,
        "2": Aciliyet.NORMAL,
        "3": Aciliyet.TAKIP,
        "4A": Aciliyet.TAKIP,
        "4B": Aciliyet.ACIL,
        "4X": Aciliyet.KRITIK,
    }

    def __init__(self):
        self.rapor_sayaci = 0

    # ----------------------------------------------------------
    # CXR Bulgu Olusturma
    # ----------------------------------------------------------
    def cxr_bulgulari_olustur(self, cxr: CXRAnaliz, esik=0.5) -> List[str]:
        """CXR patoloji olasiliklarindan bulgu cumlesi olustur."""
        bulgular = []
        # Kalite
        if cxr.kalite_skoru < 0.6:
            bulgular.append(
                f"Goruntu kalitesi dusuktur (skor: {cxr.kalite_skoru:.2f}). "
                f"Bulgular bu kisitlilik dahilinde degerlendirilmistir."
            )

        # Patolojiler (olasilik sirasina gore)
        pozitif = {k: v for k, v in cxr.patolojiler.items() if v >= esik}
        if not pozitif:
            bulgular.append("Akciger parankiminde belirgin patolojik bulgu izlenmemistir.")
            bulgular.append("Kardiyomediastinal siluet dogaldir.")
            bulgular.append("Kemik yapilar intaktir.")
            return bulgular

        for patoloji, prob in sorted(pozitif.items(), key=lambda x: -x[1]):
            tr_name = self.PATOLOJI.get(patoloji, patoloji)
            if prob >= 0.8:
                guven = "belirgin"
            elif prob >= 0.6:
                guven = "muhtemel"
            else:
                guven = "olasi"

            if patoloji == "Cardiomegaly":
                bulgular.append(
                    f"Kardiyotorasik oran artmis olup {guven} kardiyomegali ile uyumludur "
                    f"(AI skoru: {prob:.2f})."
                )
            elif patoloji == "Effusion":
                bulgular.append(
                    f"{guven.capitalize()} plevral efuzyon bulgusu izlenmektedir "
                    f"(AI skoru: {prob:.2f})."
                )
            elif patoloji == "Pneumothorax":
                bulgular.append(
                    f"{guven.capitalize()} pnomotoraks bulgusu saptanmistir "
                    f"(AI skoru: {prob:.2f}). Klinik korelasyon onerilir."
                )
            elif patoloji in ("Nodule", "Mass"):
                bulgular.append(
                    f"Akciger parankiminde {guven} noduler/kitlesel lezyon izlenmektedir "
                    f"(AI skoru: {prob:.2f}). BT ile ileri degerlendirme onerilir."
                )
            elif patoloji in ("Consolidation", "Pneumonia"):
                bulgular.append(
                    f"Akciger parankiminde {guven} konsolidasyon alani izlenmektedir "
                    f"(AI skoru: {prob:.2f}). Pnomoni on tanisi ile klinik korelasyon onerilir."
                )
            else:
                bulgular.append(
                    f"{guven.capitalize()} {tr_name.lower()} bulgusu mevcuttur "
                    f"(AI skoru: {prob:.2f})."
                )

        # Normal yapilar
        if "Cardiomegaly" not in pozitif:
            bulgular.append("Kardiyomediastinal siluet dogaldir.")
        bulgular.append("Kemik yapilar degerlendirme sinirlari dahilinde intaktir.")

        return bulgular

    # ----------------------------------------------------------
    # BT Bulgu Olusturma
    # ----------------------------------------------------------
    def bt_bulgulari_olustur(self, bt: BTAnaliz) -> List[str]:
        """BT nodul bulgularindan rapor cumlesi olustur."""
        bulgular = []

        if not bt.noduller:
            bulgular.append("Akciger parankiminde noduler lezyon saptanmamistir.")
            bulgular.append("Mediastinal ve hiler lenf nodlari dogal boyutlardadir.")
            bulgular.append("Plevral alanda patolojik bulgu izlenmemistir.")
            return bulgular

        bulgular.append(
            f"Akciger parankiminde toplam {bt.toplam_nodul} adet noduler lezyon tespit edilmistir:"
        )

        # Noduller boyuta gore sirala (buyukten kucuge)
        sorted_nodules = sorted(bt.noduller, key=lambda n: n.boyut_mm or 0, reverse=True)

        for i, nodul in enumerate(sorted_nodules, 1):
            lokasyon_tr = self.LOKASYON.get(nodul.lokasyon, nodul.lokasyon)
            boyut_str = f"{nodul.boyut_mm:.1f} mm" if nodul.boyut_mm else "boyut belirlenemedi"

            # Morfoloji bilgisi
            morfoloji = []
            shape = nodul.ozellikler.get("shape", "")
            if shape == "irregular":
                morfoloji.append("duzensiz konturlu")
            elif shape == "round":
                morfoloji.append("yuvarlak")

            density = nodul.ozellikler.get("density", "")
            if density == "solid":
                morfoloji.append("solid")
            elif density == "part-solid":
                morfoloji.append("kismi solid")
            elif density == "ground-glass":
                morfoloji.append("buzlu cam")

            morfoloji_str = ", ".join(morfoloji) if morfoloji else ""
            lr_str = f", Lung-RADS {nodul.lung_rads}" if nodul.lung_rads else ""

            if morfoloji_str:
                bulgular.append(
                    f"  {i}. {lokasyon_tr}'da {boyut_str} capinda "
                    f"{morfoloji_str} nodul{lr_str} (AI guven: {nodul.guven_skoru:.0%})."
                )
            else:
                bulgular.append(
                    f"  {i}. {lokasyon_tr}'da {boyut_str} capinda "
                    f"nodul{lr_str} (AI guven: {nodul.guven_skoru:.0%})."
                )

            # Malignite skoru yuksekse ozel not
            if nodul.malignite_skoru and nodul.malignite_skoru >= 3.5:
                bulgular.append(
                    f"     * Bu nodul yuksek malignite riski tasimaktadir "
                    f"(skor: {nodul.malignite_skoru:.1f}/5). Ileri degerlendirme onerilir."
                )

        # Ek yapilar
        bulgular.append(
            "Mediastinal ve hiler lenf nodlari degerlendirme sinirlari dahilinde dogal boyutlardadir."
        )
        bulgular.append("Plevral alanda patolojik bulgu izlenmemistir.")

        return bulgular

    # ----------------------------------------------------------
    # Izlenim Olusturma
    # ----------------------------------------------------------
    def izlenim_olustur(self, modalite: str, cxr: Optional[CXRAnaliz],
                        bt: Optional[BTAnaliz]) -> List[str]:
        """Izlenim/sonuc bolumu olustur."""
        izlenim = []

        if modalite == "CXR" and cxr:
            pozitif = {k: v for k, v in cxr.patolojiler.items() if v >= 0.5}
            if not pozitif:
                izlenim.append(
                    "Gogus rontgenograminda belirgin patolojik bulgu izlenmemistir."
                )
            else:
                for pat, prob in sorted(pozitif.items(), key=lambda x: -x[1]):
                    tr = self.PATOLOJI.get(pat, pat)
                    if prob >= 0.7:
                        izlenim.append(f"{tr} ile uyumlu bulgular mevcuttur.")
                    else:
                        izlenim.append(
                            f"{tr} acisindan kuskulu bulgular mevcuttur, "
                            f"klinik korelasyon onerilir."
                        )

        if modalite == "BT" and bt:
            if not bt.noduller:
                izlenim.append(
                    "BT incelemesinde akciger parankiminde noduler lezyon saptanmamistir."
                )
            else:
                lr = bt.genel_lung_rads
                izlenim.append(
                    f"Akciger parankiminde {bt.toplam_nodul} adet noduler lezyon saptanmistir."
                )
                izlenim.append(
                    f"En buyuk nodul {bt.en_buyuk_nodul_mm:.1f} mm capindadir."
                )
                izlenim.append(
                    f"Genel degerlendirme: Lung-RADS Kategori {lr}."
                )

                if lr in ("4A", "4B", "4X"):
                    izlenim.append(
                        "Malignite acisindan supheli bulgular mevcuttur. "
                        "Ileri degerlendirme ve/veya biyopsi onerilir."
                    )
                elif lr == "3":
                    izlenim.append(
                        "Muhtemelen benign bulgular mevcuttur. Kisa sureli takip onerilir."
                    )

        return izlenim

    # ----------------------------------------------------------
    # Aciliyet Belirleme
    # ----------------------------------------------------------
    def aciliyet_belirle(self, cxr: Optional[CXRAnaliz],
                         bt: Optional[BTAnaliz]) -> Aciliyet:
        """Bulgu ciddiyet durumuna gore aciliyet belirle."""
        aciliyet = Aciliyet.NORMAL

        if cxr:
            if cxr.patolojiler.get("Pneumothorax", 0) >= 0.6:
                return Aciliyet.KRITIK
            if cxr.patolojiler.get("Mass", 0) >= 0.7:
                aciliyet = Aciliyet.ACIL

        if bt:
            lr = bt.genel_lung_rads
            lr_aciliyet = self.LUNGRADS_ACILIYET.get(lr, Aciliyet.NORMAL)
            all_levels = list(Aciliyet)
            if all_levels.index(lr_aciliyet) > all_levels.index(aciliyet):
                aciliyet = lr_aciliyet

        return aciliyet

    # ----------------------------------------------------------
    # Takip Onerisi
    # ----------------------------------------------------------
    def takip_onerisi_olustur(self, modalite: str, bt: Optional[BTAnaliz],
                              aciliyet: Aciliyet) -> str:
        """Takip onerisi olustur."""
        if aciliyet == Aciliyet.KRITIK:
            return (
                "ACIL: Sorumlu klinisyen derhal bilgilendirilmelidir. "
                "Acil mudahale gerekebilir."
            )

        if modalite == "BT" and bt:
            lr = bt.genel_lung_rads
            return self.LUNGRADS_TAKIP.get(lr, "Klinik korelasyon onerilir.")

        if aciliyet == Aciliyet.ACIL:
            return (
                "Klinisyen bilgilendirilmeli, ileri goruntuleme/konsultasyon planlanmalidir."
            )
        elif aciliyet == Aciliyet.TAKIP:
            return "Kontrol goruntuleme ile takip onerilir."
        else:
            return "Rutin takip yeterlidir."

    # ----------------------------------------------------------
    # Ana Rapor Olusturma
    # ----------------------------------------------------------
    def rapor_olustur(self, hasta: HastaBilgisi, modalite: str,
                      cxr: Optional[CXRAnaliz] = None,
                      bt: Optional[BTAnaliz] = None) -> RaporSonucu:
        """Tam yapilandirilmis radyoloji raporu olustur."""
        self.rapor_sayaci += 1

        if not hasta.tarih:
            hasta.tarih = datetime.now().strftime("%d.%m.%Y %H:%M")

        # Teknik bilgi
        if modalite == "CXR":
            teknik = "PA (posteroanterior) gogus rontgenogrami, dijital sistem"
        else:
            teknik = "Toraks BT, IV kontrastsiz, 1 mm kesit kalinligi, dusuk doz protokol"

        # Bulgular
        bulgular = []
        if modalite == "CXR" and cxr:
            bulgular = self.cxr_bulgulari_olustur(cxr)
        elif modalite == "BT" and bt:
            bulgular = self.bt_bulgulari_olustur(bt)

        # Izlenim
        izlenim = self.izlenim_olustur(modalite, cxr, bt)

        # Aciliyet
        aciliyet = self.aciliyet_belirle(cxr, bt)

        # Lung-RADS
        lung_rads = bt.genel_lung_rads if bt else None

        # Takip
        takip = self.takip_onerisi_olustur(modalite, bt, aciliyet)

        # AI Ozet
        ai_ozet = {
            "rapor_no": int(self.rapor_sayaci),
            "modalite": modalite,
            "tarih": hasta.tarih,
            "toplam_bulgu": int(len(bulgular)),
            "aciliyet": aciliyet.value,
            "lung_rads": lung_rads,
            "pipeline_version": "AlpCAN v1.0",
        }
        if cxr:
            ai_ozet["cxr_patolojiler"] = {
                k: round(float(v), 3) for k, v in cxr.patolojiler.items() if v >= 0.3
            }
        if bt:
            ai_ozet["bt_toplam_nodul"] = int(bt.toplam_nodul)
            ai_ozet["bt_en_buyuk_mm"] = round(float(bt.en_buyuk_nodul_mm), 1)

        sonuc = RaporSonucu(
            hasta=hasta,
            modalite=Modalite.CXR.value if modalite == "CXR" else Modalite.BT.value,
            teknik=teknik,
            bulgular=bulgular,
            izlenim=izlenim,
            lung_rads=lung_rads,
            aciliyet=aciliyet.value,
            takip_onerisi=takip,
            ai_ozet=ai_ozet,
        )

        # HTML ve text rapor
        sonuc.rapor_html = self._html_olustur(sonuc)
        sonuc.rapor_text = self._text_olustur(sonuc)

        return sonuc

    # ----------------------------------------------------------
    # Duz Metin Rapor
    # ----------------------------------------------------------
    def _text_olustur(self, rapor: RaporSonucu) -> str:
        """Duz metin rapor."""
        lines = []
        lines.append("=" * 70)
        lines.append("ALpCAN - YAPILANDIRILMIS RADYOLOJI RAPORU")
        lines.append("=" * 70)
        lines.append(f"Hasta: {rapor.hasta.hasta_adi} ({rapor.hasta.hasta_id})")
        lines.append(f"Yas/Cinsiyet: {rapor.hasta.yas}/{rapor.hasta.cinsiyet}")
        lines.append(f"Protokol No: {rapor.hasta.protokol_no}")
        lines.append(f"Tarih: {rapor.hasta.tarih}")
        if rapor.hasta.klinik_bilgi:
            lines.append(f"Klinik Bilgi: {rapor.hasta.klinik_bilgi}")
        lines.append("")
        lines.append(f"INCELEME: {rapor.modalite}")
        lines.append(f"TEKNIK: {rapor.teknik}")
        lines.append("")
        lines.append("BULGULAR:")
        for b in rapor.bulgular:
            lines.append(f"  {b}")
        lines.append("")
        lines.append("IZLENIM:")
        for i, iz in enumerate(rapor.izlenim, 1):
            lines.append(f"  {i}. {iz}")
        if rapor.lung_rads:
            lines.append(f"\nLung-RADS Kategori: {rapor.lung_rads}")
        lines.append(f"\nACILIYET: {rapor.aciliyet}")
        lines.append(f"TAKIP ONERISI: {rapor.takip_onerisi}")
        lines.append("")
        lines.append("-" * 70)
        lines.append("Bu rapor AlpCAN AI destekli radyoloji sistemi tarafindan olusturulmustur.")
        lines.append("Kesin tani icin uzman radyolog degerlendirmesi gereklidir.")
        lines.append("AlpCAN v1.0 | Giresun Universitesi | TUSEB Projesi")
        lines.append("=" * 70)
        return "\n".join(lines)

    # ----------------------------------------------------------
    # HTML Rapor
    # ----------------------------------------------------------
    def _html_olustur(self, rapor: RaporSonucu) -> str:
        """HTML formatinda rapor (PDF export icin)."""
        aciliyet_renk = {
            "Normal": "#28a745",
            "Takip Gerekli": "#ffc107",
            "Acil Konsultasyon": "#fd7e14",
            "Kritik - Hemen Bildirim": "#dc3545",
        }
        renk = aciliyet_renk.get(rapor.aciliyet, "#6c757d")

        bulgular_li = ''.join(f'<li>{b}</li>' for b in rapor.bulgular)
        izlenim_li = ''.join(f'<li>{iz}</li>' for iz in rapor.izlenim)

        lr_badge = ''
        if rapor.lung_rads:
            lr_badge = (
                f'<span style="display:inline-block;background:#1a237e;color:white;'
                f'padding:5px 15px;border-radius:20px;font-weight:bold;font-size:14px;">'
                f'Lung-RADS {rapor.lung_rads}</span>'
            )

        html = f'''<!DOCTYPE html>
<html lang="tr">
<head><meta charset="UTF-8">
<style>
body {{ font-family: 'Segoe UI', Tahoma, sans-serif; max-width: 800px;
       margin: 0 auto; padding: 20px; color: #333; }}
.header {{ text-align: center; border-bottom: 3px solid #1a237e;
          padding-bottom: 15px; margin-bottom: 20px; }}
.header h1 {{ color: #1a237e; margin: 0; font-size: 20px; }}
.header h2 {{ color: #666; margin: 5px 0; font-size: 14px; font-weight: normal; }}
.patient-info {{ background: #f8f9fa; padding: 15px; border-radius: 8px;
               margin-bottom: 20px; display: grid;
               grid-template-columns: 1fr 1fr; gap: 8px; }}
.patient-info span {{ font-size: 13px; }}
.patient-info .label {{ font-weight: bold; color: #555; }}
.section {{ margin-bottom: 20px; }}
.section h3 {{ color: #1a237e; border-bottom: 1px solid #dee2e6;
             padding-bottom: 5px; font-size: 15px; }}
.section ul {{ list-style: none; padding-left: 0; }}
.section ul li {{ padding: 4px 0; font-size: 13px; line-height: 1.5; }}
.aciliyet {{ display: inline-block; background: {renk}; color: white;
           padding: 5px 15px; border-radius: 20px; font-weight: bold;
           font-size: 13px; }}
.takip {{ background: #fff3cd; padding: 12px; border-radius: 8px;
        border-left: 4px solid #ffc107; font-size: 13px; }}
.footer {{ text-align: center; color: #999; font-size: 11px;
         margin-top: 30px; border-top: 1px solid #dee2e6; padding-top: 10px; }}
</style></head>
<body>
<div class="header">
<h1>GIRESUN UNIVERSITESI TIP FAKULTESI HASTANESI</h1>
<h2>Radyoloji Anabilim Dali — AI Destekli Raporlama</h2>
</div>

<div class="patient-info">
<span><span class="label">Hasta:</span> {rapor.hasta.hasta_adi}</span>
<span><span class="label">ID:</span> {rapor.hasta.hasta_id}</span>
<span><span class="label">Yas/Cinsiyet:</span> {rapor.hasta.yas}/{rapor.hasta.cinsiyet}</span>
<span><span class="label">Protokol:</span> {rapor.hasta.protokol_no}</span>
<span><span class="label">Tarih:</span> {rapor.hasta.tarih}</span>
<span><span class="label">Inceleme:</span> {rapor.modalite}</span>
</div>

<div class="section">
<h3>TEKNIK</h3>
<p style="font-size:13px">{rapor.teknik}</p>
</div>

<div class="section">
<h3>BULGULAR</h3>
<ul>{bulgular_li}</ul>
</div>

<div class="section">
<h3>IZLENIM</h3>
<ul>{izlenim_li}</ul>
</div>

<div class="section" style="display:flex;gap:15px;align-items:center;">
{lr_badge}
<span class="aciliyet">{rapor.aciliyet}</span>
</div>

<div class="takip">
<strong>Takip Onerisi:</strong> {rapor.takip_onerisi}
</div>

<div class="footer">
<p>Bu rapor AlpCAN AI destekli radyoloji sistemi tarafindan olusturulmustur.<br>
Kesin tani icin uzman radyolog degerlendirmesi gereklidir.<br>
<strong>AlpCAN v1.0</strong> | Giresun Universitesi | TUSEB Projesi</p>
</div>
</body></html>'''
        return html


print('RaporMotoru sinifi tanimlandi.')
print('Metotlar: cxr_bulgulari_olustur, bt_bulgulari_olustur, izlenim_olustur,')
print('          aciliyet_belirle, takip_onerisi_olustur, rapor_olustur')

## 3. Demo Senaryolari

Bes farkli klinik senaryoda rapor motorunun ciktisi gosterilmektedir:

1. **Normal CXR** — Patoloji yok
2. **CXR + Pnomoni** — Konsolidasyon ve efuzyon bulgulari
3. **BT + Kucuk benign nodul** — Lung-RADS 2
4. **BT + Coklu nodul (supheli)** — Lung-RADS 4A
5. **BT + Buyuk kitle (kritik)** — Lung-RADS 4X

In [ ]:
# ============================================================
# Demo Senaryolari — 5 Farkli Klinik Durum
# ============================================================
motor = RaporMotoru()
demo_raporlar = []

# ----------------------------------------------------------
# Senaryo 1: Normal CXR
# ----------------------------------------------------------
hasta1 = HastaBilgisi(
    hasta_id="GRU-2025-001",
    hasta_adi="Ahmet Yilmaz",
    yas=45,
    cinsiyet="E",
    protokol_no="CXR-20250316-001",
    tarih="16.03.2025 09:30",
    klinik_bilgi="Rutin kontrol, sikayet yok",
)
cxr1 = CXRAnaliz(
    patolojiler={
        "Atelectasis": 0.12,
        "Cardiomegaly": 0.08,
        "Consolidation": 0.05,
        "Effusion": 0.03,
        "Nodule": 0.10,
        "Pneumonia": 0.04,
        "Pneumothorax": 0.02,
    },
    kalite_skoru=0.92,
    goruntulenebilirlik="Yeterli",
)
rapor1 = motor.rapor_olustur(hasta1, "CXR", cxr=cxr1)
demo_raporlar.append(rapor1)
print('Senaryo 1: Normal CXR')
print('=' * 60)
print(rapor1.rapor_text)
print()

# ----------------------------------------------------------
# Senaryo 2: CXR + Pnomoni
# ----------------------------------------------------------
hasta2 = HastaBilgisi(
    hasta_id="GRU-2025-002",
    hasta_adi="Fatma Demir",
    yas=62,
    cinsiyet="K",
    protokol_no="CXR-20250316-002",
    tarih="16.03.2025 10:15",
    klinik_bilgi="Ates, oksuruk, nefes darligi. Pnomoni on tanisi.",
)
cxr2 = CXRAnaliz(
    patolojiler={
        "Consolidation": 0.85,
        "Pneumonia": 0.78,
        "Effusion": 0.62,
        "Atelectasis": 0.35,
        "Cardiomegaly": 0.15,
        "Nodule": 0.08,
        "Pneumothorax": 0.03,
    },
    kalite_skoru=0.85,
    goruntulenebilirlik="Yeterli",
)
rapor2 = motor.rapor_olustur(hasta2, "CXR", cxr=cxr2)
demo_raporlar.append(rapor2)
print('Senaryo 2: CXR + Pnomoni')
print('=' * 60)
print(rapor2.rapor_text)
print()

# ----------------------------------------------------------
# Senaryo 3: BT + Kucuk Benign Nodul (Lung-RADS 2)
# ----------------------------------------------------------
hasta3 = HastaBilgisi(
    hasta_id="GRU-2025-003",
    hasta_adi="Mehmet Kaya",
    yas=55,
    cinsiyet="E",
    protokol_no="BT-20250316-001",
    tarih="16.03.2025 11:00",
    klinik_bilgi="Akciger kanseri taramasi, 30 paket-yil sigara oykusu",
)
nodul3 = Bulgu(
    id=1,
    tip="nodul",
    lokasyon="right_upper",
    boyut_mm=4.5,
    malignite_skoru=1.2,
    lung_rads="2",
    guven_skoru=0.88,
    ozellikler={"shape": "round", "density": "solid"},
)
bt3 = BTAnaliz(
    noduller=[nodul3],
    toplam_nodul=1,
    en_buyuk_nodul_mm=4.5,
    genel_lung_rads="2",
)
rapor3 = motor.rapor_olustur(hasta3, "BT", bt=bt3)
demo_raporlar.append(rapor3)
print('Senaryo 3: BT + Kucuk Benign Nodul (Lung-RADS 2)')
print('=' * 60)
print(rapor3.rapor_text)
print()

# ----------------------------------------------------------
# Senaryo 4: BT + Coklu Nodul, Supheli (Lung-RADS 4A)
# ----------------------------------------------------------
hasta4 = HastaBilgisi(
    hasta_id="GRU-2025-004",
    hasta_adi="Ayse Ozturk",
    yas=68,
    cinsiyet="K",
    protokol_no="BT-20250316-002",
    tarih="16.03.2025 14:30",
    klinik_bilgi="Kilo kaybi, kronik oksuruk. BT takip.",
    onceki_calisma="BT 15.09.2024 — 6 mm nodul sag ust lobda",
)
nodul4a = Bulgu(
    id=1,
    tip="nodul",
    lokasyon="right_upper",
    boyut_mm=9.2,
    malignite_skoru=3.1,
    lung_rads="4A",
    guven_skoru=0.92,
    ozellikler={"shape": "irregular", "density": "part-solid"},
)
nodul4b = Bulgu(
    id=2,
    tip="nodul",
    lokasyon="left_lower",
    boyut_mm=3.8,
    malignite_skoru=1.5,
    lung_rads="2",
    guven_skoru=0.85,
    ozellikler={"shape": "round", "density": "solid"},
)
nodul4c = Bulgu(
    id=3,
    tip="nodul",
    lokasyon="right_lower",
    boyut_mm=5.1,
    malignite_skoru=2.0,
    lung_rads="3",
    guven_skoru=0.79,
    ozellikler={"shape": "round", "density": "ground-glass"},
)
bt4 = BTAnaliz(
    noduller=[nodul4a, nodul4b, nodul4c],
    toplam_nodul=3,
    en_buyuk_nodul_mm=9.2,
    genel_lung_rads="4A",
)
rapor4 = motor.rapor_olustur(hasta4, "BT", bt=bt4)
demo_raporlar.append(rapor4)
print('Senaryo 4: BT + Coklu Nodul (Lung-RADS 4A)')
print('=' * 60)
print(rapor4.rapor_text)
print()

# ----------------------------------------------------------
# Senaryo 5: BT + Buyuk Kitle, Kritik (Lung-RADS 4X)
# ----------------------------------------------------------
hasta5 = HastaBilgisi(
    hasta_id="GRU-2025-005",
    hasta_adi="Ali Celik",
    yas=72,
    cinsiyet="E",
    protokol_no="BT-20250316-003",
    tarih="16.03.2025 16:00",
    klinik_bilgi="Hemoptizi, kilo kaybi, kemik agrisi. Malignite suphe.",
)
nodul5a = Bulgu(
    id=1,
    tip="kitle",
    lokasyon="left_upper",
    boyut_mm=42.0,
    malignite_skoru=4.8,
    lung_rads="4X",
    guven_skoru=0.97,
    ozellikler={"shape": "irregular", "density": "solid"},
)
nodul5b = Bulgu(
    id=2,
    tip="nodul",
    lokasyon="right_lower",
    boyut_mm=8.5,
    malignite_skoru=3.6,
    lung_rads="4A",
    guven_skoru=0.89,
    ozellikler={"shape": "irregular", "density": "solid"},
)
bt5 = BTAnaliz(
    noduller=[nodul5a, nodul5b],
    toplam_nodul=2,
    en_buyuk_nodul_mm=42.0,
    genel_lung_rads="4X",
)
rapor5 = motor.rapor_olustur(hasta5, "BT", bt=bt5)
demo_raporlar.append(rapor5)
print('Senaryo 5: BT + Buyuk Kitle, Kritik (Lung-RADS 4X)')
print('=' * 60)
print(rapor5.rapor_text)
print()

print(f'\nToplam {len(demo_raporlar)} demo rapor olusturuldu.')

In [ ]:
# ============================================================
# HTML Raporlari Dosyaya Kaydet ve Goruntule
# ============================================================
from IPython.display import display, HTML

html_dir = OUTPUT_DIR / 'html_raporlar'
html_dir.mkdir(parents=True, exist_ok=True)

for i, rapor in enumerate(demo_raporlar, 1):
    fname = html_dir / f'rapor_senaryo_{i}.html'
    with open(fname, 'w', encoding='utf-8') as f:
        f.write(rapor.rapor_html)
    fsize = fname.stat().st_size / 1024
    print(f'Kaydedildi: {fname.name} ({fsize:.1f} KB) — Aciliyet: {rapor.aciliyet}')

# Senaryo 5 (kritik) raporunu notebook icinde goster
print()
print('=' * 60)
print('Senaryo 5 HTML onizleme (kritik vaka):')
print('=' * 60)
display(HTML(demo_raporlar[4].rapor_html))

## 4. Toplu Rapor Isleme (Batch Processing)

20 simulasyon vakasi ile rapor motorunun toplu isleme kapasitesi test edilmektedir.
Rassal patoloji ve nodul verileri olusturularak istatistiksel ozet cikarilir.

In [ ]:
# ============================================================
# Toplu Rapor Isleme — 20 Simule Vaka
# ============================================================
random.seed(42)

ISIMLER_E = ['Ahmet', 'Mehmet', 'Ali', 'Mustafa', 'Hasan',
             'Huseyin', 'Ibrahim', 'Ismail', 'Yusuf', 'Omer']
ISIMLER_K = ['Fatma', 'Ayse', 'Emine', 'Hatice', 'Zeynep',
             'Elif', 'Meryem', 'Sultan', 'Havva', 'Gulsum']
SOYISIMLER = ['Yilmaz', 'Kaya', 'Demir', 'Celik', 'Ozturk',
              'Sahin', 'Arslan', 'Dogan', 'Kilic', 'Aslan',
              'Erdogan', 'Korkmaz', 'Acar', 'Kurt', 'Bulut',
              'Cengiz', 'Polat', 'Aktas', 'Tas', 'Yildiz']

CXR_PATOLOJILER = [
    'Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema',
    'Effusion', 'Emphysema', 'Fibrosis', 'Infiltration',
    'Mass', 'Nodule', 'Pleural_Thickening', 'Pneumonia', 'Pneumothorax',
]

BT_LOKASYONLAR = ['right_upper', 'right_middle', 'right_lower',
                  'left_upper', 'left_lower']
BT_SHAPES = ['round', 'irregular']
BT_DENSITIES = ['solid', 'part-solid', 'ground-glass']


def rasgele_hasta(idx):
    cinsiyet = random.choice(["E", "K"])
    if cinsiyet == "E":
        ad = random.choice(ISIMLER_E)
    else:
        ad = random.choice(ISIMLER_K)
    soyad = random.choice(SOYISIMLER)
    return HastaBilgisi(
        hasta_id=f"GRU-2025-{idx:03d}",
        hasta_adi=f"{ad} {soyad}",
        yas=random.randint(35, 80),
        cinsiyet=cinsiyet,
        protokol_no=f"BATCH-{idx:03d}",
        tarih=f"16.03.2025 {random.randint(8,17):02d}:{random.randint(0,59):02d}",
        klinik_bilgi=random.choice([
            "Rutin tarama",
            "Oksuruk, ates",
            "Nefes darligi",
            "Akciger kanseri taramasi",
            "Sigara oykusu, kontrol",
            "Kilo kaybi",
            "Gogus agrisi",
        ]),
    )


def rasgele_cxr():
    patolojiler = {}
    for pat in CXR_PATOLOJILER:
        patolojiler[pat] = round(random.random() * 0.6, 3)
    # Rastgele 0-2 patoloji yukselt
    n_pos = random.randint(0, 2)
    for pat in random.sample(CXR_PATOLOJILER, min(n_pos, len(CXR_PATOLOJILER))):
        patolojiler[pat] = round(random.uniform(0.55, 0.95), 3)
    return CXRAnaliz(
        patolojiler=patolojiler,
        kalite_skoru=round(random.uniform(0.6, 0.98), 2),
    )


def rasgele_bt():
    n_nodul = random.choices([0, 1, 2, 3], weights=[0.3, 0.35, 0.2, 0.15])[0]
    noduller = []
    max_boyut = 0.0
    max_lr = '1'
    lr_order = {'1': 0, '2': 1, '3': 2, '4A': 3, '4B': 4, '4X': 5}

    for j in range(n_nodul):
        boyut = round(random.uniform(2.0, 25.0), 1)
        malig = round(random.uniform(0.5, 5.0), 1)
        # Boyut -> Lung-RADS haritalama
        if boyut < 6:
            lr = '2'
        elif boyut < 8:
            lr = '3'
        elif boyut < 15:
            lr = '4A'
        elif boyut < 30:
            lr = '4B'
        else:
            lr = '4X'

        nodul = Bulgu(
            id=j + 1,
            tip="nodul",
            lokasyon=random.choice(BT_LOKASYONLAR),
            boyut_mm=boyut,
            malignite_skoru=malig,
            lung_rads=lr,
            guven_skoru=round(random.uniform(0.65, 0.98), 2),
            ozellikler={
                "shape": random.choice(BT_SHAPES),
                "density": random.choice(BT_DENSITIES),
            },
        )
        noduller.append(nodul)
        if boyut > max_boyut:
            max_boyut = boyut
        if lr_order.get(lr, 0) > lr_order.get(max_lr, 0):
            max_lr = lr

    return BTAnaliz(
        noduller=noduller,
        toplam_nodul=n_nodul,
        en_buyuk_nodul_mm=max_boyut,
        genel_lung_rads=max_lr if n_nodul > 0 else '1',
    )


# Toplu rapor uret
batch_motor = RaporMotoru()
batch_raporlar = []

for idx in range(100, 120):
    hasta = rasgele_hasta(idx)
    modalite = random.choice(['CXR', 'BT'])

    cxr = rasgele_cxr() if modalite == 'CXR' else None
    bt = rasgele_bt() if modalite == 'BT' else None

    rapor = batch_motor.rapor_olustur(hasta, modalite, cxr=cxr, bt=bt)
    batch_raporlar.append(rapor)

# Ozet istatistikler
print('=' * 60)
print(f'TOPLU RAPOR ISLEME OZETI — {len(batch_raporlar)} vaka')
print('=' * 60)

# Modalite dagilimi
mod_counts = Counter(r.modalite for r in batch_raporlar)
print(f'\nModalite Dagilimi:')
for mod, cnt in mod_counts.items():
    print(f'  {mod}: {cnt}')

# Aciliyet dagilimi
ac_counts = Counter(r.aciliyet for r in batch_raporlar)
print(f'\nAciliyet Dagilimi:')
for ac, cnt in sorted(ac_counts.items(), key=lambda x: list(Aciliyet).index(Aciliyet(x[0]))):
    print(f'  {ac}: {cnt}')

# Lung-RADS dagilimi (sadece BT)
lr_counts = Counter(r.lung_rads for r in batch_raporlar if r.lung_rads)
print(f'\nLung-RADS Dagilimi (BT vakalari):')
for lr, cnt in sorted(lr_counts.items()):
    print(f'  Kategori {lr}: {cnt}')

print(f'\nToplam rapor sayisi: {batch_motor.rapor_sayaci}')

In [ ]:
# ============================================================
# Gorsellesirme — Aciliyet, Lung-RADS, Patoloji Dagilimi
# ============================================================
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# --- 1. Aciliyet dagilimi (pasta grafigi) ---
ac_labels = []
ac_values = []
ac_colors_map = {
    "Normal": "#28a745",
    "Takip Gerekli": "#ffc107",
    "Acil Konsultasyon": "#fd7e14",
    "Kritik - Hemen Bildirim": "#dc3545",
}
for ac_enum in Aciliyet:
    cnt = ac_counts.get(ac_enum.value, 0)
    if cnt > 0:
        ac_labels.append(f'{ac_enum.value}\n({cnt})')
        ac_values.append(cnt)

ac_colors = [ac_colors_map.get(l.split('\n')[0], '#6c757d') for l in ac_labels]
wedges, texts, autotexts = axes[0].pie(
    ac_values, labels=ac_labels, autopct='%1.0f%%',
    colors=ac_colors, startangle=90,
    textprops={'fontsize': 9},
    wedgeprops={'edgecolor': 'white', 'linewidth': 2},
)
for at in autotexts:
    at.set_fontweight('bold')
axes[0].set_title('Aciliyet Dagilimi', fontweight='bold', fontsize=13)

# --- 2. Lung-RADS dagilimi (bar grafigi) ---
lr_order = ['1', '2', '3', '4A', '4B', '4X']
lr_bar_colors = ['#28a745', '#28a745', '#ffc107', '#fd7e14', '#dc3545', '#dc3545']
lr_vals = [lr_counts.get(cat, 0) for cat in lr_order]

bars = axes[1].bar(lr_order, lr_vals, color=lr_bar_colors, edgecolor='white', linewidth=1.5)
axes[1].set_xlabel('Lung-RADS Kategorisi')
axes[1].set_ylabel('Vaka Sayisi')
axes[1].set_title('Lung-RADS Dagilimi (BT vakalari)', fontweight='bold', fontsize=13)
for bar, val in zip(bars, lr_vals):
    if val > 0:
        axes[1].text(bar.get_x() + bar.get_width() / 2., bar.get_height() + 0.2,
                     str(val), ha='center', va='bottom', fontweight='bold')

# --- 3. En sik CXR patolojiler (yatay bar) ---
# Tum batch CXR raporlarindan pozitif patolojileri say
pat_counter = Counter()
for r in batch_raporlar:
    if "Rontgeni" in r.modalite:   # CXR
        ai = r.ai_ozet
        if "cxr_patolojiler" in ai:
            for pat_name, prob in ai["cxr_patolojiler"].items():
                if prob >= 0.5:
                    pat_counter[pat_name] += 1

if pat_counter:
    pat_names = [k for k, v in pat_counter.most_common(10)]
    pat_vals = [v for k, v in pat_counter.most_common(10)]
    y_pos = range(len(pat_names))
    axes[2].barh(y_pos, pat_vals, color='#1a237e', edgecolor='white')
    axes[2].set_yticks(y_pos)
    axes[2].set_yticklabels(pat_names, fontsize=9)
    axes[2].set_xlabel('Pozitif Vaka Sayisi')
    axes[2].invert_yaxis()
else:
    axes[2].text(0.5, 0.5, 'CXR pozitif bulgu yok',
                 ha='center', va='center', transform=axes[2].transAxes)
axes[2].set_title('En Sik CXR Patolojiler', fontweight='bold', fontsize=13)

fig.suptitle('AlpCAN RPT-1 — Batch Rapor Istatistikleri (20 vaka)',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'rpt1_batch_statistics.png', bbox_inches='tight', dpi=150)
plt.show()
print('Kaydedildi: rpt1_batch_statistics.png')

## 5. Backend Entegrasyonu

Rapor motorunu bagimsiz bir Python modulu olarak disa aktarma
ve backend API uzerinden kullanim ornegi.

In [ ]:
# ============================================================
# rapor_motoru.py — Bagimsiz Modul Olarak Disa Aktarma
# ============================================================

MODULE_CODE = '''#!/usr/bin/env python3
"""AlpCAN RPT-1: Yapilandirilmis Radyoloji Rapor Motoru.

Bu modul, AlpCAN pipeline ciktilarindan otomatik yapilandirilmis
radyoloji raporu uretir. Bagimsiz olarak kullanilabilir.

Kullanim:
    from rapor_motoru import generate_report
    result = generate_report(pipeline_json)
"""
import json
from datetime import datetime
from typing import Dict, Any, Optional


def generate_report(pipeline_json: dict) -> dict:
    """Pipeline JSON ciktisini alir, yapilandirilmis rapor uretir.

    Args:
        pipeline_json: Pipeline ciktisi. Beklenen alanlar:
            - hasta_id, hasta_adi, yas, cinsiyet, protokol_no
            - modalite: 'CXR' veya 'BT'
            - cxr_patolojiler: dict (patoloji -> olasilik)  [CXR icin]
            - bt_noduller: list of dict  [BT icin]

    Returns:
        dict: Rapor sonucu (bulgular, izlenim, aciliyet, takip, html, text)
    """
    modalite = pipeline_json.get('modalite', 'CXR')
    tarih = pipeline_json.get('tarih', datetime.now().strftime('%d.%m.%Y %H:%M'))

    # Hasta bilgisi
    hasta = {
        'hasta_id': pipeline_json.get('hasta_id', 'UNKNOWN'),
        'hasta_adi': pipeline_json.get('hasta_adi', 'Bilinmiyor'),
        'yas': int(pipeline_json.get('yas', 0)),
        'cinsiyet': pipeline_json.get('cinsiyet', '-'),
        'protokol_no': pipeline_json.get('protokol_no', '-'),
        'tarih': tarih,
        'klinik_bilgi': pipeline_json.get('klinik_bilgi', ''),
    }

    # CXR bulgu olusturma
    bulgular = []
    izlenim = []
    aciliyet = 'Normal'
    lung_rads = None
    takip = 'Rutin takip yeterlidir.'

    if modalite == 'CXR':
        patolojiler = pipeline_json.get('cxr_patolojiler', {})
        pozitif = {k: v for k, v in patolojiler.items() if v >= 0.5}
        if not pozitif:
            bulgular.append('Akciger parankiminde belirgin patolojik bulgu izlenmemistir.')
            izlenim.append('Normal gogus rontgenogrami.')
        else:
            for pat, prob in sorted(pozitif.items(), key=lambda x: -x[1]):
                bulgular.append(f'{pat}: AI skoru {prob:.2f}')
            izlenim.append(f'{len(pozitif)} adet patolojik bulgu saptanmistir.')
            if patolojiler.get('Pneumothorax', 0) >= 0.6:
                aciliyet = 'Kritik - Hemen Bildirim'
            elif patolojiler.get('Mass', 0) >= 0.7:
                aciliyet = 'Acil Konsultasyon'
            elif any(v >= 0.5 for v in patolojiler.values()):
                aciliyet = 'Takip Gerekli'

    elif modalite == 'BT':
        noduller = pipeline_json.get('bt_noduller', [])
        if not noduller:
            bulgular.append('Noduler lezyon saptanmamistir.')
            izlenim.append('Normal BT incelemesi.')
            lung_rads = '1'
        else:
            bulgular.append(f'{len(noduller)} adet nodul saptanmistir.')
            max_lr = '1'
            lr_order = {'1':0,'2':1,'3':2,'4A':3,'4B':4,'4X':5}
            for n in noduller:
                lr = n.get('lung_rads', '1')
                if lr_order.get(lr, 0) > lr_order.get(max_lr, 0):
                    max_lr = lr
                bulgular.append(
                    f"  Nodul: {n.get('lokasyon','?')} - "
                    f"{n.get('boyut_mm',0):.1f} mm - Lung-RADS {lr}"
                )
            lung_rads = max_lr
            izlenim.append(f'Lung-RADS Kategori {max_lr}.')

            lungrads_takip = {
                '1': 'Yillik tarama.',
                '2': 'Yillik tarama.',
                '3': '6 ay sonra kontrol.',
                '4A': '3 ay sonra kontrol veya PET/BT.',
                '4B': 'Biyopsi/PET-BT onerilir.',
                '4X': 'Acil cerrahi konsultasyon.',
            }
            takip = lungrads_takip.get(max_lr, 'Klinik korelasyon.')
            lungrads_aciliyet = {
                '1': 'Normal', '2': 'Normal', '3': 'Takip Gerekli',
                '4A': 'Takip Gerekli', '4B': 'Acil Konsultasyon',
                '4X': 'Kritik - Hemen Bildirim',
            }
            aciliyet = lungrads_aciliyet.get(max_lr, 'Normal')

    return {
        'hasta': hasta,
        'modalite': modalite,
        'bulgular': bulgular,
        'izlenim': izlenim,
        'aciliyet': aciliyet,
        'lung_rads': lung_rads,
        'takip_onerisi': takip,
        'pipeline_version': 'AlpCAN v1.0',
        'tarih': tarih,
    }


if __name__ == '__main__':
    # Ornek kullanim
    test_input = {
        'hasta_id': 'TEST-001',
        'hasta_adi': 'Test Hasta',
        'yas': 55,
        'cinsiyet': 'E',
        'protokol_no': 'TEST-P001',
        'modalite': 'CXR',
        'cxr_patolojiler': {
            'Consolidation': 0.82,
            'Pneumonia': 0.75,
            'Effusion': 0.45,
        },
    }
    result = generate_report(test_input)
    print(json.dumps(result, ensure_ascii=False, indent=2))
'''

# Modulu dosyaya yaz
module_path = OUTPUT_DIR / 'rapor_motoru.py'
with open(module_path, 'w', encoding='utf-8') as f:
    f.write(MODULE_CODE)
print(f'Modul kaydedildi: {module_path}')
print(f'Boyut: {module_path.stat().st_size / 1024:.1f} KB')

# Backend API entegrasyon ornegi
print()
print('=' * 60)
print('Backend Entegrasyon Ornegi:')
print('=' * 60)
print()
print('FastAPI endpoint ornegi:')
print('''
@router.post("/api/v1/reports/generate")
async def generate_radiology_report(pipeline_output: dict):
    from rapor_motoru import generate_report
    report = generate_report(pipeline_output)
    return report
''')

# Modulu test et
print('Modul testi:')
exec(MODULE_CODE)
test_input = {
    'hasta_id': 'API-TEST-001',
    'hasta_adi': 'API Test Hasta',
    'yas': 60,
    'cinsiyet': 'K',
    'protokol_no': 'API-P001',
    'modalite': 'BT',
    'bt_noduller': [
        {'lokasyon': 'right_upper', 'boyut_mm': 12.0, 'lung_rads': '4A'},
        {'lokasyon': 'left_lower', 'boyut_mm': 4.0, 'lung_rads': '2'},
    ],
}
result = generate_report(test_input)
print(json.dumps(result, ensure_ascii=False, indent=2))

In [ ]:
# ============================================================
# Tum Ciktilari Kaydet
# ============================================================

# --- 1. Demo raporlari JSON ---
def rapor_to_dict(rapor):
    """RaporSonucu dataclass'ini JSON-uyumlu dict'e cevir."""
    d = {
        'hasta': {
            'hasta_id': rapor.hasta.hasta_id,
            'hasta_adi': rapor.hasta.hasta_adi,
            'yas': int(rapor.hasta.yas),
            'cinsiyet': rapor.hasta.cinsiyet,
            'protokol_no': rapor.hasta.protokol_no,
            'tarih': rapor.hasta.tarih,
            'klinik_bilgi': rapor.hasta.klinik_bilgi,
        },
        'modalite': rapor.modalite,
        'teknik': rapor.teknik,
        'bulgular': rapor.bulgular,
        'izlenim': rapor.izlenim,
        'lung_rads': rapor.lung_rads,
        'aciliyet': rapor.aciliyet,
        'takip_onerisi': rapor.takip_onerisi,
        'ai_ozet': rapor.ai_ozet,
    }
    return d


# Demo raporlari
demo_json = [rapor_to_dict(r) for r in demo_raporlar]
demo_path = OUTPUT_DIR / 'demo_raporlar.json'
with open(demo_path, 'w', encoding='utf-8') as f:
    json.dump(demo_json, f, ensure_ascii=False, indent=2)
print(f'Kaydedildi: {demo_path.name} ({demo_path.stat().st_size / 1024:.1f} KB)')

# Batch raporlari
batch_json = [rapor_to_dict(r) for r in batch_raporlar]
batch_path = OUTPUT_DIR / 'batch_raporlar.json'
with open(batch_path, 'w', encoding='utf-8') as f:
    json.dump(batch_json, f, ensure_ascii=False, indent=2)
print(f'Kaydedildi: {batch_path.name} ({batch_path.stat().st_size / 1024:.1f} KB)')

# --- 2. Ozet CSV ---
csv_path = OUTPUT_DIR / 'rapor_ozet.csv'
with open(csv_path, 'w', encoding='utf-8', newline='') as f:
    writer = csv.writer(f)
    writer.writerow([
        'Rapor No', 'Hasta ID', 'Hasta Adi', 'Yas', 'Cinsiyet',
        'Modalite', 'Aciliyet', 'Lung-RADS', 'Bulgu Sayisi', 'Takip Onerisi',
    ])
    all_reports = demo_raporlar + batch_raporlar
    for i, r in enumerate(all_reports, 1):
        writer.writerow([
            int(i),
            r.hasta.hasta_id,
            r.hasta.hasta_adi,
            int(r.hasta.yas),
            r.hasta.cinsiyet,
            r.modalite,
            r.aciliyet,
            r.lung_rads or '-',
            int(len(r.bulgular)),
            r.takip_onerisi,
        ])
print(f'Kaydedildi: {csv_path.name} ({csv_path.stat().st_size / 1024:.1f} KB)')

# --- 3. Pipeline konfigurasyonu ---
pipeline_config = {
    "notebook": "NB14 — RPT-1 Raporlama Motoru",
    "version": "1.0",
    "pipeline": "AlpCAN RPT-1",
    "description": "Yapilandirilmis Turkce radyoloji rapor motoru",
    "capabilities": [
        "CXR patoloji raporlama",
        "BT nodul raporlama",
        "Lung-RADS kategorilendirme",
        "Aciliyet degerlendirme",
        "Takip onerisi",
        "HTML rapor uretimi",
        "JSON / CSV cikti",
    ],
    "supported_modalities": ["CXR", "BT"],
    "supported_cxr_pathologies": list(RaporMotoru.PATOLOJI.keys()),
    "lung_rads_categories": ["1", "2", "3", "4A", "4B", "4X"],
    "output_formats": ["text", "html", "json", "csv"],
    "statistics": {
        "demo_reports": int(len(demo_raporlar)),
        "batch_reports": int(len(batch_raporlar)),
        "total_reports": int(len(demo_raporlar) + len(batch_raporlar)),
    },
}
config_path = OUTPUT_DIR / 'pipeline_config.json'
with open(config_path, 'w', encoding='utf-8') as f:
    json.dump(pipeline_config, f, ensure_ascii=False, indent=2)
print(f'Kaydedildi: {config_path.name} ({config_path.stat().st_size / 1024:.1f} KB)')

# --- 4. Dosya listesi ---
print()
print('=' * 60)
print('TUM CIKTI DOSYALARI')
print('=' * 60)
output_files = sorted(OUTPUT_DIR.glob('*'))
for fpath in output_files:
    if fpath.is_file():
        size = fpath.stat().st_size / 1024
        print(f'  {fpath.name:<40s} {size:>8.1f} KB')
    elif fpath.is_dir():
        n_files = len(list(fpath.glob('*')))
        print(f'  {fpath.name + "/":<40s} ({n_files} dosya)')

In [ ]:
# ============================================================
# Final Ozet
# ============================================================
print('=' * 70)
print('AlpCAN RPT-1 — YAPILANDIRILMIS RADYOLOJI RAPOR MOTORU')
print('=' * 70)
print()
print('Tamamlanan isler:')
print('  1. RaporMotoru sinifi olusturuldu (CXR + BT destegi)')
print('  2. 5 demo senaryoda rapor uretimi dogrulandi')
print('  3. 20 batch vakada toplu isleme test edildi')
print('  4. HTML, JSON, CSV ciktilar olusturuldu')
print('  5. Backend entegrasyon icin rapor_motoru.py modulu yazildi')
print()
print('Rapor Motoru Ozellikleri:')
print('  - Turkce radyoloji terminolojisi')
print('  - ACR uyumlu rapor yapisi (Teknik, Bulgular, Izlenim)')
print('  - Lung-RADS kategorilendirme (1, 2, 3, 4A, 4B, 4X)')
print('  - Otomatik aciliyet belirleme (Normal -> Kritik)')
print('  - Takip onerisi uretimi')
print('  - HTML rapor (PDF-ready)')
print('  - JSON ve CSV export')
print()
print('Istatistikler:')
print(f'  Demo raporlar: {len(demo_raporlar)}')
print(f'  Batch raporlar: {len(batch_raporlar)}')
print(f'  Toplam uretilen rapor: {len(demo_raporlar) + len(batch_raporlar)}')
print()
print('Pipeline Entegrasyonu:')
print('  CXR Pipeline (NB02/03/05/08) -> RPT-1 -> Turkce Rapor')
print('  CT Pipeline (NB06/07/09) -> RPT-1 -> Turkce Rapor')
print()
print('Sonraki Adimlar:')
print('  - NB12 LLM tabanli rapor uretimi ile birlestirme')
print('  - Backend API entegrasyonu (/api/v1/reports/generate)')
print('  - PDF export modulu ekleme')
print('  - Radyolog geri bildirim mekanizmasi')
print()
print('=' * 70)
print('AlpCAN v1.0 | Giresun Universitesi | TUSEB Projesi')
print('=' * 70)